# 04 — cross-stage synthesis

**triplet-phase-lock**

This notebook synthesizes the full pipeline:

- **Pi (expand):** what expands?
- **π (extend):** what extends?
- **Π (resist):** what resists?

We compare clean, weakly perturbed, strongly perturbed, and noisy sequence families using a shared set of metrics across all three stages.


## Setup

The goal here is not to introduce new definitions, but to bring the previous notebooks into one view.

We keep the same triplet construction:

\\[\nT_k = (N_k, N_{k+1}, N_{k+2})\n\\]\n
and compare:

- raw sequence expansion
- local extension magnitude
- directional continuity and directional change
- resistance scores and strict resistance masks


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['figure.figsize'] = (8, 4)
plt.rcParams['axes.grid'] = True


## Shared helpers


In [ ]:
def sequence_n(k):
    k = np.asarray(k, dtype=float)
    return 24.0 * k - 25.0


def sequence_n_perturbed(k, amplitude=8.0, period=6.0):
    k = np.asarray(k, dtype=float)
    return sequence_n(k) + amplitude * np.sin(k / period)


def sequence_n_noisy(k, amplitude=40.0, seed=7):
    k = np.asarray(k, dtype=float)
    rng = np.random.default_rng(seed)
    return sequence_n(k) + rng.normal(loc=0.0, scale=amplitude, size=len(k))


def build_triplets_from_values(vals):
    vals = np.asarray(vals, dtype=float)
    return np.stack([vals[:-2], vals[1:-1], vals[2:]], axis=1)


def local_differences(x):
    if len(x) < 2:
        return np.array([], dtype=float)
    return np.linalg.norm(np.diff(x, axis=0), axis=1)


def component_differences(x):
    if len(x) < 2:
        return np.empty((0, x.shape[1]), dtype=float)
    return np.diff(x, axis=0)


def normalize_rows(x, eps=1e-12):
    norms = np.linalg.norm(x, axis=1, keepdims=True)
    return x / np.maximum(norms, eps)


def cosine_between_steps(x):
    diffs = np.diff(x, axis=0)
    dirs = normalize_rows(diffs)
    if len(dirs) < 2:
        return np.array([], dtype=float)
    return np.sum(dirs[:-1] * dirs[1:], axis=1)


def direction_change_norm(x):
    diffs = np.diff(x, axis=0)
    dirs = normalize_rows(diffs)
    if len(dirs) < 2:
        return np.array([], dtype=float)
    return np.linalg.norm(dirs[1:] - dirs[:-1], axis=1)


def cosine_scores(x, reference):
    x_norm = normalize_rows(x)
    r = np.asarray(reference, dtype=float)
    r = r / max(np.linalg.norm(r), 1e-12)
    return x_norm @ r


def resistance_mask_from_scores(scores, threshold):
    return scores >= threshold


def resistance_ratio(mask):
    mask = np.asarray(mask, dtype=bool)
    if mask.size == 0:
        return 0.0
    return float(mask.mean())


## Generate sequence families


In [ ]:
num_steps = 80
k = np.arange(1, num_steps + 3)

families = {
    'clean': sequence_n(k),
    'weak': sequence_n_perturbed(k, amplitude=8.0, period=6.0),
    'strong': sequence_n_perturbed(k, amplitude=20.0, period=4.0),
    'noisy': sequence_n_noisy(k, amplitude=40.0, seed=7),
}

triplets = {name: build_triplets_from_values(vals) for name, vals in families.items()}

for name in families:
    print(name, 'sequence shape:', families[name].shape, '| triplet shape:', triplets[name].shape)


## Stage 1 — expand

A first comparison of raw sequence growth across the four families.


In [ ]:
plt.figure()
for name, vals in families.items():
    plt.plot(k, vals, marker='o', label=name)
plt.xlabel('k')
plt.ylabel('value')
plt.title('Expand stage: raw sequence comparison')
plt.legend()
plt.tight_layout()
plt.show()


## Stage 2 — extend

Compute the three extension metrics used in Notebook 02:

- local triplet extension magnitude
- directional continuity
- directional change


In [ ]:
delta = {name: local_differences(T) for name, T in triplets.items()}
dircos = {name: cosine_between_steps(T) for name, T in triplets.items()}
dirchg = {name: direction_change_norm(T) for name, T in triplets.items()}

for name in triplets:
    print(f"{name:>6} | Δ range: {delta[name].min():.12f} to {delta[name].max():.12f} | "
          f"dir-cos range: {dircos[name].min():.12f} to {dircos[name].max():.12f} | "
          f"dir-change range: {dirchg[name].min():.12f} to {dirchg[name].max():.12f}")


In [ ]:
idx_delta = np.arange(1, len(delta['clean']) + 1)

plt.figure()
for name in ['clean', 'weak', 'strong', 'noisy']:
    plt.plot(idx_delta, delta[name], marker='o', label=name)
plt.xlabel('k')
plt.ylabel(r'$\Delta_k$')
plt.title('Extend stage: local triplet extension magnitude')
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
idx_dircos = np.arange(1, len(dircos['clean']) + 1)

plt.figure()
plt.ylim(0.9998, 1.00001)
for name in ['clean', 'weak', 'strong', 'noisy']:
    plt.plot(idx_dircos, dircos[name], marker='o', label=name)
plt.xlabel('k')
plt.ylabel('cosine between steps')
plt.title('Extend stage: directional continuity')
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
idx_dirchg = np.arange(1, len(dirchg['clean']) + 1)

plt.figure()
for name in ['clean', 'weak', 'strong', 'noisy']:
    plt.plot(idx_dirchg, dirchg[name], marker='o', label=name)
plt.xlabel('k')
plt.ylabel(r'$\|\hat d_{k+1} - \hat d_k\|$')
plt.title('Extend stage: directional change between steps')
plt.legend()
plt.tight_layout()
plt.show()


## Stage 3 — resist

Reuse the Notebook 03 logic with the empirical clean reference and compare broad vs strict thresholds.


In [ ]:
reference_clean = normalize_rows(triplets['clean']).mean(axis=0)
reference_clean = reference_clean / np.linalg.norm(reference_clean)

threshold_45 = 1.0 / np.sqrt(1.0**2 + 1.0**2)
threshold_strict = 0.9995

scores = {name: cosine_scores(T, reference_clean) for name, T in triplets.items()}
mask45 = {name: resistance_mask_from_scores(scores[name], threshold_45) for name in triplets}
maskS = {name: resistance_mask_from_scores(scores[name], threshold_strict) for name in triplets}

for name in triplets:
    print(f"{name:>6} | score range: {scores[name].min():.12f} to {scores[name].max():.12f} | "
          f"45° ratio: {resistance_ratio(mask45[name]):.6f} | strict ratio: {resistance_ratio(maskS[name]):.6f}")


In [ ]:
idx_scores = np.arange(1, len(scores['clean']) + 1)

plt.figure()
for name in ['clean', 'weak', 'strong', 'noisy']:
    plt.plot(idx_scores, scores[name], marker='o', label=name)
plt.axhline(threshold_45, linestyle='--', label='45° threshold')
plt.axhline(threshold_strict, linestyle='--', label='strict threshold')
plt.xlabel('k')
plt.ylabel(r'$\cos\theta_k$')
plt.title('Resist stage: clean-reference resistance scores')
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(8, 7), sharex=True)

for name in ['clean', 'weak', 'strong', 'noisy']:
    axes[0].plot(idx_scores, mask45[name].astype(float), marker='o', label=name)
axes[0].set_ylabel('accepted / rejected')
axes[0].set_title('Resist stage: masks under the 45° threshold')
axes[0].legend()

for name in ['clean', 'weak', 'strong', 'noisy']:
    axes[1].plot(idx_scores, maskS[name].astype(float), marker='o', label=name)
axes[1].set_xlabel('k')
axes[1].set_ylabel('accepted / rejected')
axes[1].set_title('Resist stage: masks under the strict threshold')
axes[1].legend()

plt.tight_layout()
plt.show()


## Cross-stage summary table

Collect one representative metric from each stage:

- **expand:** final sequence value
- **extend:** mean local extension magnitude and mean directional change
- **resist:** mean cosine score and strict resistance ratio


In [ ]:
summary_rows = []

for name in ['clean', 'weak', 'strong', 'noisy']:
    row = {
        'family': name,
        'expand_final_value': float(families[name][-1]),
        'extend_mean_delta': float(delta[name].mean()),
        'extend_mean_dir_change': float(dirchg[name].mean()),
        'resist_mean_score': float(scores[name].mean()),
        'resist_strict_ratio': float(resistance_ratio(maskS[name])),
    }
    summary_rows.append(row)

print('Cross-stage summary:')
for row in summary_rows:
    print(
        f"{row['family']:>6} | "
        f"expand final={row['expand_final_value']:.4f} | "
        f"mean Δ={row['extend_mean_delta']:.6f} | "
        f"mean dir-change={row['extend_mean_dir_change']:.6f} | "
        f"mean score={row['resist_mean_score']:.6f} | "
        f"strict ratio={row['resist_strict_ratio']:.6f}"
    )


## Cross-stage synthesis figure

This stacks one representative view from each stage into a single figure.


In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(8, 10), sharex=False)

# Expand
for name in ['clean', 'weak', 'strong', 'noisy']:
    axes[0].plot(k, families[name], marker='o', label=name)
axes[0].set_xlabel('k')
axes[0].set_ylabel('value')
axes[0].set_title('Expand: raw sequences')
axes[0].legend()

# Extend
for name in ['clean', 'weak', 'strong', 'noisy']:
    axes[1].plot(idx_dirchg, dirchg[name], marker='o', label=name)
axes[1].set_xlabel('k')
axes[1].set_ylabel(r'$\|\hat d_{k+1} - \hat d_k\|$')
axes[1].set_title('Extend: directional change')
axes[1].legend()

# Resist
for name in ['clean', 'weak', 'strong', 'noisy']:
    axes[2].plot(idx_scores, maskS[name].astype(float), marker='o', label=name)
axes[2].set_xlabel('k')
axes[2].set_ylabel('accepted / rejected')
axes[2].set_title('Resist: strict-threshold masks')
axes[2].legend()

plt.tight_layout()
plt.show()


## Interpretation

Across the full pipeline:

- **clean** remains stable in all three stages
- **weak** preserves strong global structure with only small local drift
- **strong** preserves global growth but shows larger directional drift and more strict-threshold failures
- **noisy** still follows the broad arithmetic trend, but local coherence drops and strict-threshold acceptance becomes intermittent

This makes the core repo claim visible:

> resistance depends not only on alignment, but on alignment relative to a chosen threshold scale.


## Summary

This notebook completed the synthesis stage:

- summarized expand → extend → resist
- compared all four sequence families across the same metrics
- showed how stricter thresholds reveal resistance structure
- connected local drift to resistance failure
